# Sending the news of the newest release to the world

GitHub's webhook delivers a release to the inbox; a rule from a file notices it; a
workflow checks which platforms have credentials, fills a template of the release's
news, shuffles it a little per platform, and posts it with the links of the release,
the package and the documentation.

Here it stays on `dryrun`, and the webhook is one message put in by hand. For real it needs
`amoorion serve` behind ngrok and a vault with accounts.


In [1]:
from pathlib import Path

import random

from amoorion import Amoorion, Plugins

from amoorion.models import Message

from amoorion.utilities import post_from_markdown

from amoorion.workflows import EXECUTE, log, step

# The rule and the template are the files of resources/, wherever this runs from.
ROOT = next(
    folder
    for folder in [Path.cwd(), *Path.cwd().parents]
    if (folder / "resources" / "rulesets").is_dir()
)

amo = Amoorion()

In [2]:
@amo.workflow(
    "hello-world",
    rules=ROOT / "resources" / "rulesets" / "world.rule.yml",
)
def hello_world_posting(
    amo,
    inputs,
):
    """The release's news, to every platform with credentials."""

    platforms = []

    @step
    def available():
        """Platforms with credentials, that can post."""

        platforms.extend(
            str(plug)
            for plug in Plugins
            if plug.infoflags["creds"] and plug.infoflags.get("post", True)
        )

        log(f"platforms: {platforms}")

        return platforms != []

    @step
    def links():
        """Each platform gets the same news, its tags in another order."""

        post, _ = post_from_markdown(
            ROOT / "resources" / "templates" / "release.template.md",
            {
                "project": "amoorion",
                "version": inputs.get(
                    "version",
                    "1.0",
                ),
                "release_url": "https://github.com/Sarverott/amoorion/releases/latest",
                "package_url": "https://pypi.org/project/amoorion/",
                "docs_url": "https://amoorion.readthedocs.io",
            },
        )

        answers = {}

        for platform in platforms:
            tags = random.sample(
                post.tags,
                len(post.tags),
            )

            [answer] = amo.publish(
                title=post.title,
                text=post.text,
                tags=tags,
                links=post.links,
                to=[platform],
            )

            answers[platform] = answer["message"]["text"]

        return answers

    return EXECUTE(
        available,
        links,
    ).serial()

In [3]:
# What GitHub's webhook would have delivered.
amo.store.keep(
    [
        Message(
            platform="hook:github",
            text="Sarverott/amoorion: release published: v1.0",
            id="release-1",
        )
    ]
)

[run] = amo.react()

print(
    *run.logs,
    sep="\n",
)

print(run.artifact["links"]["dryrun"])

available: begins
platforms: ['dryrun']
available: done
links: begins
links: done
amoorion 1.0 is out

amoorion 1.0 is out. Release, package and documentation are linked below.
